# ClinicalBERT Medication NER Evaluation on MACCROBAT

This notebook evaluates `pabRomero/BioClinicalBERT-full-finetuned-ner-pablo` against all 400 MACCROBAT documents.

The schemas are different, so evaluation is limited to compatible medication categories. MACCROBAT `Medication` is mapped to model `Drug`; model `Strength` and both schemas' `Dosage` are evaluated as `Dosage`; `Frequency` and `Duration` are direct matches. Unsupported categories are excluded from scoring.

## Load the Model

In [1]:
from __future__ import annotations

import numpy as np
import torch
from datasets import load_dataset
from seqeval.metrics import accuracy_score, f1_score, precision_score, recall_score
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)

MODEL_NAME = "pabRomero/BioClinicalBERT-full-finetuned-ner-pablo"
IGNORE_LABEL_ID = -100

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME)

print(f"Loaded {MODEL_NAME}.")
print("Evaluation device:", "cuda" if torch.cuda.is_available() else "cpu")
print("Model labels:", model.config.id2label)

python(30822) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(30823) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded pabRomero/BioClinicalBERT-full-finetuned-ner-pablo.
Evaluation device: cpu
Model labels: {0: 'O', 1: 'B-Drug', 2: 'I-Drug', 3: 'B-Reason', 4: 'I-Reason', 5: 'B-Route', 6: 'I-Route', 7: 'B-Strength', 8: 'I-Strength', 9: 'B-Form', 10: 'I-Form', 11: 'B-Dosage', 12: 'I-Dosage', 13: 'B-Frequency', 14: 'I-Frequency', 15: 'B-Duration', 16: 'I-Duration', 17: 'B-ADE', 18: 'I-ADE'}


## Load MACCROBAT

In [2]:
maccrobat_evaluation = load_dataset(
    "ktgiahieu/maccrobat2018_2020",
    split="train",
)

print("Evaluation documents:", len(maccrobat_evaluation))
print("Dataset columns:", maccrobat_evaluation.column_names)

Evaluation documents: 400
Dataset columns: ['tokens', 'tags']


## Define the Shared Label Schema

Only categories that can be reasonably aligned between the two schemas are scored. Other gold and predicted categories become `O` for this scoped evaluation.

In [3]:
GOLD_ENTITY_MAP = {
    "Medication": "Drug",
    "Dosage": "Dosage",
    "Frequency": "Frequency",
    "Duration": "Duration",
}

PREDICTED_ENTITY_MAP = {
    "Drug": "Drug",
    "Strength": "Dosage",
    "Dosage": "Dosage",
    "Frequency": "Frequency",
    "Duration": "Duration",
}

SCOPED_LABELS = {
    "O",
    "B-Drug", "I-Drug",
    "B-Dosage", "I-Dosage",
    "B-Frequency", "I-Frequency",
    "B-Duration", "I-Duration",
}

missing_model_labels = sorted(SCOPED_LABELS - set(model.config.label2id))
print("Scoped evaluation labels:", sorted(SCOPED_LABELS))
print("Missing from model:", missing_model_labels)

if missing_model_labels:
    raise ValueError("The model is missing labels required by the shared schema.")

Scoped evaluation labels: ['B-Dosage', 'B-Drug', 'B-Duration', 'B-Frequency', 'I-Dosage', 'I-Drug', 'I-Duration', 'I-Frequency', 'O']
Missing from model: []


## Tokenize and Align MACCROBAT Labels

In [4]:
def map_gold_label(label: str) -> str:
    if label == "O":
        return "O"

    bio_prefix, entity_type = label.split("-", maxsplit=1)
    mapped_type = GOLD_ENTITY_MAP.get(entity_type)
    return f"{bio_prefix}-{mapped_type}" if mapped_type else "O"


def tokenize_and_align_maccrobat(examples):
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=512,
        stride=0,
        return_overflowing_tokens=True,
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    aligned_labels = []

    for chunk_index, sample_index in enumerate(sample_mapping):
        word_ids = tokenized.word_ids(batch_index=chunk_index)
        document_tags = examples["tags"][sample_index]
        chunk_labels = []
        previous_word_id = None

        for word_id in word_ids:
            if word_id is None or word_id == previous_word_id:
                chunk_labels.append(IGNORE_LABEL_ID)
            else:
                mapped_label = map_gold_label(document_tags[word_id])
                chunk_labels.append(model.config.label2id[mapped_label])
            previous_word_id = word_id

        aligned_labels.append(chunk_labels)

    tokenized["labels"] = aligned_labels
    return tokenized


tokenized_evaluation = maccrobat_evaluation.map(
    tokenize_and_align_maccrobat,
    batched=True,
    remove_columns=maccrobat_evaluation.column_names,
    desc="Tokenizing MACCROBAT medication evaluation set",
)

print("Evaluation documents:", len(maccrobat_evaluation))
print("Tokenized chunks:", len(tokenized_evaluation))

Evaluation documents: 400
Tokenized chunks: 760


## Calculate Scoped Medication Metrics

Model predictions outside the shared schema are excluded. `Strength` predictions are normalized to `Dosage` to match MACCROBAT's broader dosage category.

In [5]:
def map_predicted_label(label: str) -> str:
    if label == "O":
        return "O"

    bio_prefix, entity_type = label.split("-", maxsplit=1)
    mapped_type = PREDICTED_ENTITY_MAP.get(entity_type)
    return f"{bio_prefix}-{mapped_type}" if mapped_type else "O"


def compute_ner_metrics(evaluation_result):
    logits, label_ids = evaluation_result
    predicted_ids = np.argmax(logits, axis=-1)
    predictions = []
    references = []

    for predicted_row, label_row in zip(predicted_ids, label_ids):
        kept_predictions = []
        kept_references = []

        for predicted_id, label_id in zip(predicted_row, label_row):
            if label_id == IGNORE_LABEL_ID:
                continue
            predicted_label = model.config.id2label[int(predicted_id)]
            kept_predictions.append(map_predicted_label(predicted_label))
            kept_references.append(model.config.id2label[int(label_id)])

        predictions.append(kept_predictions)
        references.append(kept_references)

    return {
        "precision": precision_score(references, predictions, zero_division=0),
        "recall": recall_score(references, predictions, zero_division=0),
        "f1": f1_score(references, predictions, zero_division=0),
        "accuracy": accuracy_score(references, predictions),
    }

In [7]:
evaluation_arguments = TrainingArguments(
    output_dir="outputs/clinicalbert-medication-maccrobat-evaluation",
    per_device_eval_batch_size=8,
    report_to="none",
)

evaluator = Trainer(
    model=model,
    args=evaluation_arguments,
    eval_dataset=tokenized_evaluation,
    processing_class=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_ner_metrics,
)

evaluation_metrics = evaluator.evaluate()
evaluation_metrics

Training Loss,Validation Loss,Step,Precision,Recall,F1,Accuracy
No log,0.185752,0,0.345751,0.547401,0.423812,0.974380


{'eval_loss': 0.1857515573501587,
 'eval_precision': 0.3457505518763797,
 'eval_recall': 0.5474006116207951,
 'eval_f1': 0.423811939793675,
 'eval_accuracy': 0.9743804696510084}